# Lab 2.3 — Prompt Caching and Stateful Conversations

**Duration:** ~10 min · **Level:** L300 · **Lab 2 of 4 — part 3/3**

This notebook covers three advanced Mantle features:

1. **Prompt-prefix caching** on Chat Completions via `extra_body.cache_salt`.
2. **Stateful conversations** on the **Responses API** via
   `previous_response_id` (an OpenAI Responses feature that Mantle supports
   faithfully — not a Mantle-only primitive).
3. **Prompt caching via `cache_control`** on the Anthropic Messages API.

These features collectively drive down latency and cost for multi-turn,
long-context, or repeated-prompt workloads.


In [1]:
import os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-west-2")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

from src.common.mantle import (
    openai_client, anthropic_client,
    GPT_OSS_120B, GPT_5_4, CLAUDE_OPUS_47,
)

client = openai_client()                  # OSS models on /v1
frontier = openai_client(frontier=True)   # frontier models (gpt-5.4) on /openai/v1
anth = anthropic_client()

## 1. Prompt-prefix caching on Chat Completions

Mantle's internal inference engine shares a vLLM-class prefix cache across
requests. You can *signal* cache affinity with `extra_body.cache_salt` —
two requests with the same prompt prefix **and** the same `cache_salt` are
eligible to hit the same cache.

To *see* the effect, we'll run the same 2 KB system prompt twice and
compare **time-to-first-token**.


In [2]:
LONG_SYSTEM = (
    "You are an enterprise risk analyst. Respond using exactly three bullets. "
    * 80
)
print(f"prefix length: {len(LONG_SYSTEM)} chars (~{len(LONG_SYSTEM)//4} tokens rough)")

def timed_call(salt: str) -> float | None:
    t0 = time.time()
    ttft = None
    stream = client.chat.completions.create(
        model=GPT_OSS_120B,
        messages=[
            {"role": "system", "content": LONG_SYSTEM},
            {"role": "user",   "content": "List three risks of migrating LLM workloads."},
        ],
        max_tokens=80,
        stream=True,
        extra_body={"cache_salt": salt},
    )
    # Drain the whole stream — we need the generator to finish cleanly so the
    # underlying HTTP connection is released. We only record the *first* TTFT.
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and ttft is None:
            ttft = time.time() - t0
    return ttft

cold = timed_call("workshop-demo-v1")    # likely a miss
warm = timed_call("workshop-demo-v1")    # likely a hit — same salt, same prefix

def fmt(v): return f"{v:.3f}s" if v is not None else "n/a"
print(f"cold TTFT: {fmt(cold)}")
print(f"warm TTFT: {fmt(warm)}")
if cold and warm:
    print(f"warm is {cold/warm:.2f}x faster at TTFT")
else:
    print("measurement failed — one of the streams produced no content")


prefix length: 5840 chars (~1460 tokens rough)
cold TTFT: 1.091s
warm TTFT: n/a
measurement failed — one of the streams produced no content


### What cache-affinity buys you

- TTFT speedups of 2× – 10× are typical when the cached prefix is large
  (system prompts, few-shot examples, RAG contexts).
- It does **not** reduce *billed* input tokens — you still pay for the
  context (on most models; verify per model card).
- The cache is *best-effort*. Don't build correctness assumptions on top of
  it — it can evict at any time.

**When to use:** a system prompt + few-shot block that you reuse across
many user questions. Pick a `cache_salt` keyed off the prefix content hash
so updates invalidate the cache naturally.


## 2. Stateful conversations with the Responses API

The **Responses API** is Mantle's native stateful primitive. Pass
`previous_response_id=<id from the last call>` and Mantle reconstructs the
conversation server-side — you don't have to send history back.

This cuts ingress bytes on long conversations and makes it easier to
maintain reasoning chains (the model keeps its own thinking).


In [3]:
r1 = client.responses.create(
    model=GPT_OSS_120B,
    input="My name is Alice and I like the color teal.",
    instructions="You are a helpful assistant who remembers user facts.",
    max_output_tokens=60,
    store=True,                        # persist state server-side
)
print("turn 1:", r1.output_text.strip())
print("id    :", r1.id)


turn 1: 
id    : resp_syet5kka3o5yjltsqf62drw4yae4tb3sfu47cucb7jzczkbhckwa


In [4]:
r2 = client.responses.create(
    model=GPT_OSS_120B,
    input="What color do I like, and what is my name?",
    previous_response_id=r1.id,        # continues the thread server-side
    max_output_tokens=80,
)
print("turn 2:", r2.output_text.strip())


turn 2: You like the color **teal**, and your name is **Alice**.


### 2a. Stateful `gpt-5.4` over the Responses API

`openai.gpt-5.4` is a frontier model exposed through Mantle's OpenAI-passthrough
surface (`/openai/v1`), so we use the dedicated `frontier` client created above.
The OSS models above use the native `/v1` surface. The `previous_response_id`
mechanism is **identical** on both — no client-side history, no Mantle-only
primitives.

Two things differ in practice from the OSS models:

- **Reasoning effort.** `gpt-5.4` accepts `reasoning={"effort": ...}` so you can
  trade latency for depth. The server keeps the reasoning chain alive across
  turns when you thread `previous_response_id`.
- **Richer state.** Because reasoning is preserved server-side, follow-up turns
  can build on earlier *thinking*, not just earlier text.

In [5]:
# Turn 1 — establish context. store=True persists state (incl. reasoning) server-side.
g1 = frontier.responses.create(
    model=GPT_5_4,
    input=(
        "I'm planning a migration of 12 microservices from Bedrock Runtime "
        "Converse to Mantle. Three of them use tool-calling loops."
    ),
    instructions="You are a senior migration architect. Be concise and concrete.",
    reasoning={"effort": "medium"},
    max_output_tokens=300,
    store=True,
)
print("turn 1:", g1.output_text.strip())
print("id    :", g1.id)

turn 1: 
id    : resp_342a64nbekp54qi5mdidnost2fpq3yniteghackki4ig3iwhjfmq


In [6]:
# Turn 2 — no history resent; only the previous response id. The model still
# "knows" the 12 services / 3 tool-loops context AND its own earlier reasoning.
g2 = frontier.responses.create(
    model=GPT_5_4,
    input="Which ones should I migrate first, and why?",
    previous_response_id=g1.id,
    reasoning={"effort": "medium"},
    max_output_tokens=400,
)
print("turn 2:", g2.output_text.strip())

# Turn 3 — keep threading off the latest id to extend the chain.
g3 = frontier.responses.create(
    model=GPT_5_4,
    input="Give me a one-line rollback plan for the tool-loop services.",
    previous_response_id=g2.id,
    max_output_tokens=200,
)
print("\nturn 3:", g3.output_text.strip())

turn 2: Migrate in this order:

## 1) First wave: 2–3 **non-tool** services
Choose services with:
- low business criticality
- low traffic
- simple request/response patterns
- short prompts, no streaming if possible
- good existing test coverage

### Why first
- Fastest way to validate the **Mantle client, auth, retries, timeouts, observability, and prompt compatibility**
- Lowest blast radius if output quality shifts
- Lets you tune defaults before touching complex flows

### Best candidates
Pick services that are:
- synchronous
- stateless
- no tool calls
- no strict JSON/schema dependency unless Mantle compatibility is already confirmed

---

## 2) Second wave: remaining **non-tool** services
After the first 2–3 are stable, migrate the other 6–7 simple services.

### Why second
- You’ll already have:
  - a hardened adapter
  - response normalization
  - quality baselines
  - rollback playbook
- These migrations become mostly mechanical

### Suggested order inside this wave
Prioritiz

### When to use Responses statefulness

- **Use it** for multi-turn agents where the server can reliably keep the
  thread alive (chat UIs, Slack bots, IDE assistants).
- **Avoid it** when you need reproducibility / audit — history is opaque.
  Prefer explicit client-managed `messages=[…]` for compliance workloads.
- **Use `store=False`** if you want the request to remain stateless but
  still *look* like a Responses call (same SDK shape as Chat Completions).


## 3. Anthropic prompt caching (`cache_control`)

On the Anthropic Messages path, caching is **explicit**. You mark a content
block with `cache_control: {"type": "ephemeral"}` and Claude caches that
prefix for ~5 minutes. You get:

- ~90% discount on *input tokens that hit the cache*.
- Lower TTFT.
- Verifiable cache hit / miss counts in `usage.cache_read_input_tokens`
  and `usage.cache_creation_input_tokens`.


In [7]:
# Claude Opus 4.5+/Sonnet 4.5/Haiku 4.5 require >= 4096 tokens per cache
# checkpoint (older Opus 4 / 3.7 Sonnet only need 1024). If the prefix is
# below the minimum the request still succeeds but the cache is silently
# skipped — usage.cache_creation_input_tokens stays at 0.
# 800 reps of "This is company policy text. " is ~5.6K tokens, comfortably
# above the 4096 threshold for Opus 4.7.
LONG_CONTEXT = (
    "# Internal policy document (excerpt)\n\n"
    + "This is company policy text. " * 800          # ~5.6K tokens
)

def claude_call(question: str):
    return anth.messages.create(
        model=CLAUDE_OPUS_47,
        max_tokens=200,
        system=[
            {
                "type": "text",
                "text": LONG_CONTEXT,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": question}],
    )

first  = claude_call("Summarize the policy in one sentence.")
second = claude_call("Are there any exceptions in the policy?")

def report(resp, label):
    u = resp.usage
    read   = getattr(u, "cache_read_input_tokens", 0) or 0
    create = getattr(u, "cache_creation_input_tokens", 0) or 0
    print(f"{label}: input={u.input_tokens}  cache_read={read}  cache_create={create}  output={u.output_tokens}")

report(first,  "first ")
report(second, "second")
print("\nSecond call should have cache_read_input_tokens ~= LONG_CONTEXT size.")


first : input=23  cache_read=0  cache_create=5617  output=1
second: input=23  cache_read=0  cache_create=5617  output=2

Second call should have cache_read_input_tokens ~= LONG_CONTEXT size.


## 4. Recap — which cache for which workload

| Workload | Best Mantle primitive | Why |
|---|---|---|
| Long system prompt reused across thousands of user questions (RAG / few-shot) | Chat Completions `cache_salt` | Covers all OpenAI-compat models; no token math. |
| Multi-turn chat / agentic thread | Responses API `previous_response_id` | Zero-byte context passing; server keeps state. |
| Long policy / document context on Claude | Anthropic `cache_control: ephemeral` | Large discount on input tokens; observable. |

**Next:** Lab 3 migrates a real Bedrock Runtime Converse tool-loop to
Mantle step-by-step, then benchmarks TTFT / tokens-per-second side by side.
